# 第 07 课 · 圆柱尾流的诊断、力指标与可信度检查

> **学习模式：**独立编写 JAX CFD 求解器 | VS Code / Jupyter | 从公式到可运行代码

**本课目标**：学会记录探针、信号与近似受力指标，理解什么样的数据可以用于验证，什么不能。

**前置知识**：第 06 课能够稳定运行。

**操作约定**：先逐单元执行，再尝试“实验与练习”；若改变了函数定义或网格设置，请重启内核并执行全部单元格。JAX 第一次调用包含编译时间。

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax import lax
jax.config.update("jax_enable_x64", True)
print("JAX:", jax.__version__, "devices:", jax.devices())

## 1. 这节课要避免的常见误区

**流场图好看 ≠ CFD 求解器可信。** 用于毕设时至少应报告网格、时间步长、散度残差、阻力/升力计算方法、边界条件、参考验证和计算资源。

对于圆柱，经典量包括：

$$C_D=\frac{F_x}{\frac12\rho U_{\rm ref}^2D},\quad
C_L=\frac{F_y}{\frac12\rho U_{\rm ref}^2D},\quad
St=\frac{fD}{U_{\rm ref}}.$$

**但本课程的平滑体积惩罚 + 周期域，暂不提供经过验证的圆柱表面应力积分。** 以下仅记录 `penalty proxy`，用于排查模型和比较参数变化，不能直接作为文献 $C_D$。

In [ ]:
def grid(nx=64,ny=32,Lx=8.,Ly=4.):
 dx,dy=Lx/nx,Ly/ny
 x=jnp.arange(nx)*dx; y=jnp.arange(ny)*dy
 X,Y=jnp.meshgrid(x,y,indexing='ij')
 return dx,dy,X,Y

In [ ]:
def div(u,v,dx,dy):
 return (u-jnp.roll(u,1,0))/dx+(v-jnp.roll(v,1,1))/dy

In [ ]:
def grad(p,dx,dy):
 return (jnp.roll(p,-1,0)-p)/dx,(jnp.roll(p,-1,1)-p)/dy

In [ ]:
def lap(q,dx,dy):
 return (jnp.roll(q,-1,0)+jnp.roll(q,1,0)-2*q)/(dx*dx)+(jnp.roll(q,-1,1)+jnp.roll(q,1,1)-2*q)/(dy*dy)

In [ ]:
def symbol(nx,ny,dx,dy):
 kx=2*jnp.pi*jnp.fft.fftfreq(nx,d=dx)[:,None]
 ky=2*jnp.pi*jnp.fft.fftfreq(ny,d=dy)[None,:]
 s=-4*jnp.sin(kx*dx/2)**2/dx**2-4*jnp.sin(ky*dy/2)**2/dy**2
 return jnp.where(s==0,1.,s)

In [ ]:
def solve_poisson(rhs,s):
 rhs_hat=jnp.fft.fftn(rhs)
 p=jnp.fft.ifftn(rhs_hat/s).real
 return p-p.mean()

In [ ]:
def project(u,v,dx,dy,s):
 p=solve_poisson(div(u,v,dx,dy),s)
 px,py=grad(p,dx,dy)
 return u-px,v-py,p

In [ ]:
def advection(u,v,dx,dy):
 v_u=0.25*(v+jnp.roll(v,-1,0)+jnp.roll(v,1,1)+jnp.roll(jnp.roll(v,-1,0),1,1))
 u_v=0.25*(u+jnp.roll(u,1,0)+jnp.roll(u,-1,1)+jnp.roll(jnp.roll(u,1,0),-1,1))
 dux=(jnp.roll(u,-1,0)-jnp.roll(u,1,0))/(2*dx)
 duy=(jnp.roll(u,-1,1)-jnp.roll(u,1,1))/(2*dy)
 dvx=(jnp.roll(v,-1,0)-jnp.roll(v,1,0))/(2*dx)
 dvy=(jnp.roll(v,-1,1)-jnp.roll(v,1,1))/(2*dy)
 return u*dux+v_u*duy,u_v*dvx+v*dvy

In [ ]:
def geometry(nx,ny,dx,dy,cx=2,cy=2,r=.5,eps=None):
 eps=0.8*min(dx,dy) if eps is None else eps
 xu=(jnp.arange(nx)[:,None]+.5)*dx; yu=jnp.arange(ny)[None,:]*dy
 xv=jnp.arange(nx)[:,None]*dx; yv=(jnp.arange(ny)[None,:]+.5)*dy
 # simple nonperiodic distance OK when body away from edges
 ru=jnp.sqrt((xu-cx)**2+(yu-cy)**2)
 rv=jnp.sqrt((xv-cx)**2+(yv-cy)**2)
 masku=.5*(1-jnp.tanh((ru-r)/eps))
 maskv=.5*(1-jnp.tanh((rv-r)/eps))
 return masku,maskv

In [ ]:
def step_navier(u,v,dx,dy,dt,nu,s,masku=None,maskv=None,eta=.02,targetU=0.,body_v=0.,gain=0.):
 def rhs(u,v):
  advu,advv=advection(u,v,dx,dy)
  force=gain*(targetU-jnp.mean(u))
  return -advu+nu*lap(u,dx,dy)+force,-advv+nu*lap(v,dx,dy)
 def filter_and_project(u,v):
  if masku is not None:
   u=u/(1.+dt/eta*masku)
   v=(v+dt/eta*maskv*body_v)/(1.+dt/eta*maskv)
  un,vn,p=project(u,v,dx,dy,s)
  return un,vn,p
 du,dv=rhs(u,v)
 u1,v1,p1=filter_and_project(u+dt*du,v+dt*dv)
 du1,dv1=rhs(u1,v1)
 u2,v2,p=filter_and_project(0.5*(u+u1+dt*du1),0.5*(v+v1+dt*dv1))
 return u2,v2,u1,v1,p

In [ ]:
nx,ny,Lx,Ly=80,40,8.,4.
dx,dy,X,Y=grid(nx,ny,Lx,Ly)
S=symbol(nx,ny,dx,dy)
cx,cy,R=2.,2.,.5
mu,mv=geometry(nx,ny,dx,dy,cx,cy,R)
Uref,Re=1.,80.
nu=Uref*(2*R)/Re;dt=.016;eta=.03
u=jnp.ones((nx,ny))*Uref
xv=jnp.arange(nx)[:,None]*dx;yv=(jnp.arange(ny)[None,:]+.5)*dy
v=.002*jnp.exp(-((xv-2.6)**2+(yv-2.25)**2)/.25)
u,v,_=project(u,v,dx,dy,S)
probe=(int(4.0/dx),int(2.5/dy))

## 2. 使用 `lax.scan` 保存时间序列

`lax.scan` 能把一系列状态更新写成可 JIT 编译的循环，允许只保存低维诊断信息，而不是存储每一个时间步的整个流场。以下 `proxy_x`、`proxy_y` **不是标准力系数**，只是掩膜区域的惩罚反馈监测量。

In [ ]:
@jax.jit
def record_block(state):
 def one_step(carry,_):
  u,v=carry
  un,vn,*_=step_navier(u,v,dx,dy,dt,nu,S,mu,mv,eta=eta,targetU=Uref,gain=.5)
  proxy_x=jnp.sum(mu*un/eta)*dx*dy
  proxy_y=jnp.sum(mv*vn/eta)*dx*dy
  signal=vn[probe]
  diag=jnp.max(jnp.abs(div(un,vn,dx,dy)))
  return (un,vn),jnp.array([proxy_x,proxy_y,signal,diag])
 return lax.scan(one_step,state,None,length=300)

(state,history)=record_block((u,v))
u,v=state
h=np.asarray(history)
print('force proxy (last):',h[-1,:2],'max |div|:',h[:,3].max())
assert np.isfinite(h).all()

In [ ]:
tt=np.arange(len(h))*dt
fig,axs=plt.subplots(3,1,figsize=(9,7),constrained_layout=True)
for ax,ix,name in zip(axs,[0,1,2],['Drag-related penalty proxy','Lift-related penalty proxy','Probe transverse velocity']):
 ax.plot(tt,h[:,ix]);ax.set_ylabel(name);ax.grid(alpha=.3)
axs[-1].set_xlabel('simulation time')
plt.show()

## 3. 频谱与 Strouhal 数的严谨条件

只有在出现**充分发展的、近似稳定周期信号**后，才能根据探针的主频估计 $St$。上面的短时序列只是练习 `scan` 与诊断，不应直接把最大 FFT 峰值称为“圆柱涡脱落频率”。

此外，提取气动力需要更严格的受力公式：

$$\mathbf F=\oint_{\partial B}\left[-p\mathbf I+\mu_f(\nabla\mathbf u+\nabla\mathbf u^{\mathsf T})\right]\mathbf n\,\mathrm ds,$$

这里 $\mu_f=\rho\nu$ 是动力黏度。因为本教学求解器使用平滑的圆柱掩膜而非真实边界面，尚未实现准确的表面应力积分。

**课程小结**：数值实验必须区分“学习原型”“验证数据”与“可用于科研论文的结果”。

## 4. 作业

1. 将 `length=300` 改为 `1000`，比较探针信号的前后差别；记得检查 CFL 和计算耗时。
2. 对探针信号进行截断、去均值和 FFT，自己写出频谱代码。若没有形成稳定周期，不要报告 $St$。
3. 在 `80×40` 与 `96×48` 网格上比较 `proxy_x`，思考为什么它不具备可靠网格收敛性。

**本课通过条件**：输出时间序列，并清楚解释为何当前力指标不能直接作为准确的 $C_D/C_L$。